
# Exploracion BIS bilateral

Este cuaderno es una exploracion separada del flujo unilateral.

La idea es mirar primero los archivos bilaterales reales antes de adaptar el codigo principal. En concreto se revisa:

1. que sesiones candidatas bilaterales existen;
2. que estructura tiene el `.spa` bilateral;
3. que canales crudos aparecen en el `.r4a`;
4. que columnas del `.spa` son informativas y cuales son sentinelas;
5. si se puede reconstruir una DSA exploratoria desde los cuatro canales crudos;
6. que bloqueos quedan antes de comparar contra una matriz `.f_a` bilateral.

Las decisiones e hipotesis de esta exploracion se registran en:

`docs/analysis/bilateral_bis_exploration_decisions.md`


In [1]:

# ============================================================
# Imports y rutas base
# ============================================================

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

# El notebook esta dentro de /notebooks. Si se ejecuta desde ahi, la raiz
# del proyecto es la carpeta padre. Si se ejecuta desde la raiz, se conserva.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

NOTEBOOKS_DIR = ROOT / "notebooks"
DATA_DIR = ROOT / "data"
DECISION_LOG = ROOT / "docs" / "analysis" / "bilateral_bis_exploration_decisions.md"

sys.path.insert(0, str(NOTEBOOKS_DIR))

import funciones_aux as fau
import funciones_dsa as fun_dsa
import funciones_dsa_bilateral as fun_dsa_b

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("ROOT:", ROOT)
print("Decision log:", DECISION_LOG)


ROOT: C:\Users\vicbr\Documents\TFG Cris
Decision log: C:\Users\vicbr\Documents\TFG Cris\docs\analysis\bilateral_bis_exploration_decisions.md



## 1. Inventario de archivos candidatos

Para el flujo bilateral nos interesan especialmente los archivos `.r4a`, porque contienen cuatro canales crudos intercalados. Tambien se revisan `.spa`, `.f_a` y `.m_a` para saber que referencias procesadas existen.


In [2]:

# ============================================================
# Inventario de sesiones BIS disponibles
# ============================================================

extensiones_interes = {".spa", ".r4a", ".r2a", ".f_a", ".m_a", ".e_a", ".ara", ".h_a", ".o_a", ".t_a"}

filas = []
for path in DATA_DIR.rglob("*"):
    if not path.is_file():
        continue
    if "__MACOSX" in str(path):
        continue
    if path.suffix.lower() not in extensiones_interes:
        continue

    filas.append({
        "sesion_dir": str(path.parent.relative_to(ROOT)),
        "archivo": path.name,
        "extension": path.suffix.lower(),
        "bytes": path.stat().st_size,
    })

df_archivos = pd.DataFrame(filas).sort_values(["sesion_dir", "archivo"]).reset_index(drop=True)

df_sesiones = (
    df_archivos
    .pivot_table(index="sesion_dir", columns="extension", values="bytes", aggfunc="sum", fill_value=0)
    .reset_index()
)

for ext in sorted(extensiones_interes):
    if ext not in df_sesiones.columns:
        df_sesiones[ext] = 0

df_sesiones["tiene_r4a"] = df_sesiones[".r4a"] > 0
df_sesiones["tiene_spa"] = df_sesiones[".spa"] > 0
df_sesiones["fa_no_vacio"] = df_sesiones[".f_a"] > 0

display(df_sesiones[["sesion_dir", "tiene_spa", "tiene_r4a", "fa_no_vacio", ".spa", ".r4a", ".f_a", ".m_a"]])


extension,sesion_dir,tiene_spa,tiene_r4a,fa_no_vacio,.spa,.r4a,.f_a,.m_a
0,data\data_bis_advanced\M-TA6m-03041035\DH03041035,True,False,True,1058379,0,683054,4517
1,data\data_bis_advanced\M-TA6m-03041035_2\DH030...,True,False,True,1058379,0,683054,4517
2,data\data_bis_antiguo\H03041352_SNB443048\H030...,True,False,False,335872,0,0,2616
3,data\data_bis_antiguo\H03041355_SNB443048\H030...,True,False,False,10381883,0,0,73799
4,data\data_bis_antiguo\L03041419\L03041419,True,False,False,80838,0,0,684
5,data\data_bis_antiguo\L04141330,True,True,False,5110448,5898240,0,7467



## 2. Seleccion del primer caso bilateral

En este repositorio, el primer candidato claro para bilateral es `L04141330`, porque tiene un `.r4a` con cuatro canales. El `.f_a` existe pero pesa 0 bytes, asi que de momento no se puede hacer la comparacion matriz-a-matriz equivalente al flujo unilateral.


In [3]:

# ============================================================
# Caso bilateral elegido para la primera exploracion
# ============================================================

CASO_BILATERAL = ROOT / "data" / "data_bis_antiguo" / "L04141330" / "L04141330"

rutas_caso = {
    "spa": CASO_BILATERAL.with_suffix(".spa"),
    "r4a": CASO_BILATERAL.with_suffix(".r4a"),
    "f_a": CASO_BILATERAL.with_suffix(".f_a"),
    "m_a": CASO_BILATERAL.with_suffix(".m_a"),
}

resumen_rutas = pd.DataFrame([
    {
        "tipo": tipo,
        "ruta": str(ruta.relative_to(ROOT)),
        "existe": ruta.exists(),
        "bytes": ruta.stat().st_size if ruta.exists() else 0,
    }
    for tipo, ruta in rutas_caso.items()
])

display(resumen_rutas)

if rutas_caso["f_a"].exists() and rutas_caso["f_a"].stat().st_size == 0:
    print("Aviso: el .f_a bilateral existe pero esta vacio. No se puede validar DSA contra .f_a en este caso.")


,tipo,ruta,existe,bytes
0,spa,data\data_bis_antiguo\L04141330\L04141330.spa,True,5110448
1,r4a,data\data_bis_antiguo\L04141330\L04141330.r4a,True,5898240
2,f_a,data\data_bis_antiguo\L04141330\L04141330.f_a,True,0
3,m_a,data\data_bis_antiguo\L04141330\L04141330.m_a,True,7467


Aviso: el .f_a bilateral existe pero esta vacio. No se puede validar DSA contra .f_a en este caso.



## 3. Estructura del `.spa` bilateral

Aqui se revisa la estructura real del `.spa`. La idea es no asumir que todos los sufijos son validos: primero se mide cuantas columnas hay por sufijo y despues se mira que variables contienen datos utiles.


In [4]:

# ============================================================
# Lectura del .spa y conteo de sufijos
# ============================================================

df_spa_raw = fau.procesar_spa(rutas_caso["spa"])

suffix_counts = {"base": 0, "_2": 0, "_3": 0, "_4": 0}
for col in df_spa_raw.columns:
    if col.endswith("_2"):
        suffix_counts["_2"] += 1
    elif col.endswith("_3"):
        suffix_counts["_3"] += 1
    elif col.endswith("_4"):
        suffix_counts["_4"] += 1
    else:
        suffix_counts["base"] += 1

print("Shape .spa raw:", df_spa_raw.shape)
print("Columnas por sufijo:", suffix_counts)
print("Primeras columnas:")
print(list(df_spa_raw.columns[:35]))
print("Ultimas columnas:")
print(list(df_spa_raw.columns[-20:]))


Shape .spa raw: (5766, 97)
Columnas por sufijo: {'base': 37, '_2': 20, '_3': 20, '_4': 20}
Primeras columnas:
['Time', 'SpSmooth', 'BiSmooth', 'LoFilter', 'NotFiltr', 'HiFilter', 'PIC_ID', 'SR12', 'SEF08', 'MEDFRQ08', 'BISBIT00', 'DB13U01', 'DB11U04', 'B34U05', 'TOTPOW08', 'EMGLOW01', 'SQI10', 'IMPEDNCE', 'ARTF2', 'BURST', 'ST', 'ASYM09', 'SBIS01', 'SEMG01', 'RESVR0', 'RESVR1', 'RESVR2', 'SR12_2', 'SEF08_2', 'MEDFRQ08_2', 'BISBIT00_2', 'DB13U01_2', 'DB11U04_2', 'B34U05_2', 'TOTPOW08_2']
Ultimas columnas:
['IMPEDNCE_4', 'ARTF2_4', 'BURST_4', 'ST_4', 'ASYM09_4', 'SBIS01_4', 'SEMG01_4', 'RESVR0_4', 'RESVR1_4', 'RESVR2_4', 'C1POSIMP', 'C1NEGIMP', 'GNDIMP', 'C2POSIMP', 'C2NEGIMP', 'C3POSIMP', 'C3NEGIMP', 'C4POSIMP', 'C4NEGIMP', 'BILBITS']



## 4. Sentinelas y columnas informativas

En el unilateral ya vimos que los valores sentinela son claves. En el bilateral esto es todavia mas importante, porque puede haber columnas repetidas que en realidad no contienen informacion util.

Se revisan variables clinicas/espectrales importantes por los cuatro sufijos:

- `base`, que corresponde a la columna sin sufijo;
- `_2`;
- `_3`;
- `_4`.


In [5]:

# ============================================================
# Tabla de validez por variable y sufijo del .spa
# ============================================================

sentinelas = [-327.7, -3276.0, -3276.8, -3276, -32767, -32768, 32768]
variables_interes = [
    "SEF08", "MEDFRQ08", "SQI10", "BISBIT00", "TOTPOW08",
    "EMGLOW01", "SR12", "DB13U01", "DB11U04", "B34U05",
    "ARTF2", "BURST", "ST", "ASYM09", "IMPEDNCE"
]

sufijos_spa = [
    ("base", ""),
    ("_2", "_2"),
    ("_3", "_3"),
    ("_4", "_4"),
]

filas_validez = []
for variable in variables_interes:
    for etiqueta_sufijo, sufijo in sufijos_spa:
        col = variable + sufijo
        if col not in df_spa_raw.columns:
            continue

        serie_raw = pd.to_numeric(df_spa_raw[col], errors="coerce")
        mask_sentinela = serie_raw.isin(sentinelas)
        serie_limpia = serie_raw.replace(sentinelas, np.nan)

        filas_validez.append({
            "variable": variable,
            "sufijo_spa": etiqueta_sufijo,
            "columna": col,
            "nan_pct": serie_raw.isna().mean() * 100,
            "sentinel_pct": mask_sentinela.mean() * 100,
            "valid_pct": serie_limpia.notna().mean() * 100,
            "min": serie_limpia.min(),
            "max": serie_limpia.max(),
            "n_unicos": serie_limpia.nunique(dropna=True),
        })

df_validez_spa = pd.DataFrame(filas_validez)

display(df_validez_spa.round(2))

print("Resumen compacto de porcentaje valido:")
display(
    df_validez_spa
    .pivot_table(index="variable", columns="sufijo_spa", values="valid_pct", aggfunc="first")
    .round(1)
)


,variable,sufijo_spa,columna,nan_pct,sentinel_pct,valid_pct,min,max,n_unicos
0,SEF08,base,SEF08,0.00,0.00,100.00,10.1,25.5,154
1,SEF08,_2,SEF08_2,0.00,100.00,0.00,NaN,NaN,0
2,SEF08,_3,SEF08_3,0.00,0.00,100.00,9.4,25.9,163
3,SEF08,_4,SEF08_4,0.00,100.00,0.00,NaN,NaN,0
4,MEDFRQ08,base,MEDFRQ08,0.00,0.00,100.00,2.1,9.3,62
5,MEDFRQ08,_2,MEDFRQ08_2,0.00,100.00,0.00,NaN,NaN,0
6,MEDFRQ08,_3,MEDFRQ08_3,0.00,0.00,100.00,2.1,9.3,63
7,MEDFRQ08,_4,MEDFRQ08_4,0.00,100.00,0.00,NaN,NaN,0
8,SQI10,base,SQI10,0.00,0.00,100.00,35.7,89.7,73
9,SQI10,_2,SQI10_2,0.00,0.00,100.00,62.2,100.0,58


Resumen compacto de porcentaje valido:


sufijo_spa,_2,_3,_4,base
variable,,,,
ARTF2,99.9,99.9,100.0,100.0
ASYM09,0.0,0.0,0.0,100.0
B34U05,0.0,0.0,0.0,0.0
BISBIT00,100.0,1.1,100.0,0.6
BURST,0.0,0.0,0.0,0.0
DB11U04,0.0,0.0,0.0,0.0
DB13U01,0.0,100.0,0.0,100.0
EMGLOW01,0.0,100.0,0.0,100.0
IMPEDNCE,98.7,98.7,98.7,100.0



## 5. Lectura limpia con el helper bilateral existente

El helper actual `limpiar_spa_bilateral` separa columnas en `izq` y `der`. Esta exploracion no lo toma como verdad final: se usa para ver que salida produce y se compara contra la tabla de sentinelas anterior.

Decision abierta: no cambiar todavia este helper hasta confirmar como deben mapearse exactamente las columnas procesadas del `.spa` frente a los canales fisicos del `.r4a`.


In [6]:

# ============================================================
# Limpieza bilateral existente
# ============================================================

df_spa_bilat = fun_dsa_b.limpiar_spa_bilateral(df_spa_raw)

print("Shape .spa bilateral limpio:", df_spa_bilat.shape)
print("Columnas limpias:")
print(list(df_spa_bilat.columns))

display(df_spa_bilat.head(5))


Shape .spa bilateral limpio: (5766, 31)
Columnas limpias:
['Time', 'SR12_izq', 'SEF08_izq', 'MEDFRQ08_izq', 'BISBIT00_izq', 'DB13U01_izq', 'DB11U04_izq', 'B34U05_izq', 'TOTPOW08_izq', 'EMGLOW01_izq', 'SQI10_izq', 'IMPEDNCE_izq', 'ARTF2_izq', 'BURST_izq', 'ST_izq', 'SR12_der', 'SEF08_der', 'MEDFRQ08_der', 'BISBIT00_der', 'DB13U01_der', 'DB11U04_der', 'B34U05_der', 'TOTPOW08_der', 'EMGLOW01_der', 'SQI10_der', 'IMPEDNCE_der', 'ARTF2_der', 'BURST_der', 'ST_der', 'ASYM09', 'modo_spa']


,Time,SR12_izq,SEF08_izq,MEDFRQ08_izq,BISBIT00_izq,DB13U01_izq,DB11U04_izq,B34U05_izq,TOTPOW08_izq,EMGLOW01_izq,SQI10_izq,IMPEDNCE_izq,ARTF2_izq,BURST_izq,ST_izq,SR12_der,SEF08_der,MEDFRQ08_der,BISBIT00_der,DB13U01_der,DB11U04_der,B34U05_der,TOTPOW08_der,EMGLOW01_der,SQI10_der,IMPEDNCE_der,ARTF2_der,BURST_der,ST_der,ASYM09,modo_spa
0,2026-04-14 13:30:44,0.0,13.4,2.2,NaN,97.7,NaN,NaN,71.8,47.1,50.0,2.5,0.0,NaN,0,0.0,13.8,2.6,NaN,91.0,NaN,NaN,69.9,46.1,53.8,1.4,0.0,NaN,0,63.1,bilateral
1,2026-04-14 13:30:45,0.0,13.8,2.2,NaN,97.7,NaN,NaN,71.7,47.0,51.3,2.3,0.0,NaN,0,0.0,13.7,2.6,NaN,92.8,NaN,NaN,69.8,46.1,55.1,2.1,80.0,NaN,0,62.4,bilateral
2,2026-04-14 13:30:46,0.0,14.3,2.3,NaN,97.7,NaN,NaN,71.5,47.2,52.6,2.4,0.0,NaN,0,0.0,13.9,2.6,NaN,95.5,NaN,NaN,69.7,46.4,56.4,2.7,0.0,NaN,0,62.1,bilateral
3,2026-04-14 13:30:47,0.0,14.6,2.4,NaN,97.7,NaN,NaN,71.3,47.2,53.8,2.2,0.0,NaN,0,0.0,14.1,2.7,NaN,97.2,NaN,NaN,69.6,46.1,57.7,2.4,0.0,NaN,0,62.0,bilateral
4,2026-04-14 13:30:48,0.0,14.6,2.4,NaN,97.7,NaN,NaN,71.3,47.3,54.5,2.1,200.0,NaN,0,0.0,14.2,2.7,NaN,97.5,NaN,NaN,69.5,46.1,59.0,2.9,80.0,NaN,0,62.3,bilateral



## 6. Lectura del `.r4a`: cuatro canales crudos

El `.r4a` contiene cuatro canales intercalados. Segun la hipotesis de trabajo aportada por usuario:

- canales 1 y 3 pertenecen al hemisferio izquierdo;
- canales 2 y 4 pertenecen al hemisferio derecho.

Aqui solo se inventarian estadisticas basicas. Todavia no se decide cual de ellos debe compararse contra cada columna procesada del `.spa`.


In [7]:

# ============================================================
# Lectura del .r4a y estadisticas por canal crudo
# ============================================================

df_eeg_r4a = fun_dsa_b.leer_r4a(rutas_caso["r4a"], fs=128, escala_uv=0.0511)
canales_uv = ["canal_1_uV", "canal_2_uV", "canal_3_uV", "canal_4_uV"]

print("Shape .r4a:", df_eeg_r4a.shape)
print("Duracion aproximada r4a (s):", round(df_eeg_r4a["tiempo_s"].iloc[-1], 2))
print("Filas .spa:", len(df_spa_raw))

display(df_eeg_r4a[canales_uv].describe().round(3))

mapeo_canales_crudos = pd.DataFrame([
    {"canal_crudo": 1, "columna": "canal_1_uV", "hemisferio_hipotesis": "izquierdo"},
    {"canal_crudo": 3, "columna": "canal_3_uV", "hemisferio_hipotesis": "izquierdo"},
    {"canal_crudo": 2, "columna": "canal_2_uV", "hemisferio_hipotesis": "derecho"},
    {"canal_crudo": 4, "columna": "canal_4_uV", "hemisferio_hipotesis": "derecho"},
])

display(mapeo_canales_crudos)


Shape .r4a: (737280, 9)
Duracion aproximada r4a (s): 5759.99
Filas .spa: 5766


,canal_1_uV,canal_2_uV,canal_3_uV,canal_4_uV
count,737280.000,737280.000,737280.000,737280.000
mean,-155.117,-141.881,-143.543,-145.624
std,64.538,34.437,54.433,31.010
min,-1672.963,-1116.790,-1651.450,-1078.006
25%,-178.850,-155.600,-165.666,-156.519
50%,-154.731,-140.934,-143.438,-144.358
75%,-130.356,-126.370,-121.209,-132.758
max,1672.299,1354.559,1659.370,1363.144


,canal_crudo,columna,hemisferio_hipotesis
0,1,canal_1_uV,izquierdo
1,3,canal_3_uV,izquierdo
2,2,canal_2_uV,derecho
3,4,canal_4_uV,derecho


## 7. Reconstruccion DSA bilateral de mejor esfuerzo desde `.r4a`

Como el `.f_a`/GLF bilateral esta vacio, aqui no se intenta una comparacion matriz-a-matriz contra una referencia exportada por el monitor.

La estrategia es adaptar lo que mejor funciono en el unilateral:

1. reconstruir cada canal crudo con Welch de 1 segundo;
2. mantener la potencia en escala lineal;
3. combinar canales por hemisferio fisico segun la hipotesis aportada:
   - izquierda = canales 1 y 3;
   - derecha = canales 2 y 4;
4. convertir a dB solo al final, para visualizar;
5. conservar tambien una matriz global de los cuatro canales.

Esto no sustituye una validacion contra `.f_a`, pero nos da una reconstruccion trazable, reproducible y alineada con el flujo unilateral.

In [8]:

# ============================================================
# Reconstruccion bilateral reutilizando el helper nuevo
# ============================================================

fs = 128
ventana_seg = 1
paso_seg = 1

# Este helper concentra la logica para no repetirla a mano en cada prueba:
# - reconstruye canales 1, 2, 3 y 4;
# - combina 1/3 como izquierdo y 2/4 como derecho;
# - calcula una matriz global de los cuatro canales;
# - deja versiones lineales y versiones en dB.
resultado_recon_bilateral = fun_dsa_b.reconstruir_dsa_bilateral_desde_r4a(
    df_eeg=df_eeg_r4a,
    fs=fs,
    ventana_seg=ventana_seg,
    paso_seg=paso_seg,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="inicio"
)

frecuencias_recon = resultado_recon_bilateral["frecuencias"]
cols_freq_recon = resultado_recon_bilateral["cols_freq"]
potencias_bilaterales = resultado_recon_bilateral["potencias"]
dsa_bilateral_db = resultado_recon_bilateral["dsa_db"]

# Variables de compatibilidad con las celdas exploratorias posteriores.
potencias_por_canal = {
    canal: potencias_bilaterales[f"canal_{canal}"]
    for canal in [1, 2, 3, 4]
}
potencia_izq_13 = potencias_bilaterales["izq_13"]
potencia_der_24 = potencias_bilaterales["der_24"]
potencia_global_1234 = potencias_bilaterales["global_1234"]

resumen_recon_bilateral = fun_dsa_b.resumir_reconstruccion_bilateral(
    resultado_recon_bilateral
)

print("Frecuencias reconstruidas:", float(frecuencias_recon.min()), "a", float(frecuencias_recon.max()), "Hz")
print("Numero de frecuencias:", len(frecuencias_recon))
print("Filas temporales reconstruidas:", len(resultado_recon_bilateral["tiempo_s"]))
print("Representaciones disponibles:", list(potencias_bilaterales.keys()))

display(resumen_recon_bilateral.round(4))


Frecuencias reconstruidas: 0.5 a 30.0 Hz
Numero de frecuencias: 60
Filas temporales reconstruidas: 5760
Representaciones disponibles: ['canal_1', 'canal_2', 'canal_3', 'canal_4', 'izq_13', 'der_24', 'global_1234']


,matriz,filas_tiempo,n_frecuencias,pot_min,pot_p50,pot_p95,pot_max,db_min,db_p50,db_p95,db_max
0,canal_1,5760,60,0.0000,0.9129,131.7265,66261.7543,16.1031,79.6042,101.1967,128.2126
1,canal_2,5760,60,0.0000,1.8531,44.7087,44611.8394,20.0960,82.6790,96.5039,126.4945
2,canal_3,5760,60,0.0000,0.8656,110.9201,46282.8975,23.7651,79.3730,100.4501,126.6542
3,canal_4,5760,60,0.0000,0.6900,39.3152,45684.3540,21.4543,78.3886,95.9456,126.5977
4,izq_13,5760,60,0.0001,1.0247,138.6675,45605.4790,41.6747,80.1059,101.4197,126.5902
5,der_24,5760,60,0.0009,1.5022,45.8314,45119.3354,49.4204,81.7674,96.6116,126.5436
6,global_1234,5760,60,0.0013,1.4348,96.1226,45362.4072,51.0405,81.5679,99.8283,126.5670


## 7.1. Suavizado bilateral respetando una mascara de calidad

En el unilateral el suavizado temporal mejoro mucho el parecido visual con la matriz del monitor, pero alli se podia optimizar contra `.f_a`.

Aqui no hay `.f_a` valido, asi que no se optimizan ni suavizado ni shift. Para no inventar una alineacion temporal sin referencia, se deja una salida suavizada de uso exploratorio:

- se usa un suavizado causal de 25 segundos, que fue competitivo en el unilateral;
- el suavizado no cruza filas marcadas como mala calidad;
- la mascara se calcula con las columnas `SQI10*` disponibles en el `.spa`.

Esta salida es para visualizacion y revision cualitativa, no para afirmar una mejora numerica contra una referencia inexistente.

In [9]:

# ============================================================
# Suavizado temporal bilateral respetando mascara de calidad
# ============================================================

umbral_sqi_bilateral = 15
suavizado_bilateral_s = 25

# Usamos todas las columnas SQI disponibles para construir una mascara conservadora.
# Si cualquier SQI disponible cae por debajo del umbral, ese segundo se considera no valido.
sqi_cols_bilateral = [c for c in ["SQI10", "SQI10_2", "SQI10_3", "SQI10_4"] if c in df_spa_raw.columns]

sentinelas_sqi_bilateral = [-327.7, -3276.0, -3276.8, -3276, -32767, -32768, 32768]

def serie_spa_limpia_bilateral(columna):
    """
    Limpia una columna del .spa solo para construir la mascara de calidad.

    Se define aqui para que esta seccion sea autonoma y no dependa de
    funciones declaradas en celdas posteriores.
    """
    return (
        pd.to_numeric(df_spa_raw[columna], errors="coerce")
        .replace(sentinelas_sqi_bilateral, np.nan)
        .to_numpy(dtype=float)
    )

sqi_limpias = []
for col in sqi_cols_bilateral:
    sqi_limpias.append(serie_spa_limpia_bilateral(col))

if len(sqi_limpias):
    sqi_stack_bilateral = np.vstack(sqi_limpias)
    sqi_min_bilateral = np.nanmin(sqi_stack_bilateral, axis=0)
    mask_no_valida_bilateral = pd.Series(
        (~np.isfinite(sqi_min_bilateral)) | (sqi_min_bilateral < umbral_sqi_bilateral)
    )
else:
    # Si en algun archivo faltase SQI, no bloqueamos la reconstruccion:
    # simplemente no se aplica mascara de calidad.
    sqi_min_bilateral = np.full(len(resultado_recon_bilateral["tiempo_s"]), np.nan)
    mask_no_valida_bilateral = pd.Series(False, index=range(len(resultado_recon_bilateral["tiempo_s"])))

# La reconstruccion dura 5760 s y el .spa puede tener unas filas extra; recortamos a la longitud comun.
n_mask = min(len(mask_no_valida_bilateral), len(resultado_recon_bilateral["tiempo_s"]))
mask_no_valida_bilateral = mask_no_valida_bilateral.iloc[:n_mask].reset_index(drop=True)

# Se suavizan las representaciones que tienen interpretacion bilateral directa.
dsa_bilateral_db_suavizada = {}
for nombre in ["izq_13", "der_24", "global_1234"]:
    dsa_base = dsa_bilateral_db[nombre].iloc[:n_mask].reset_index(drop=True)
    dsa_bilateral_db_suavizada[nombre] = fun_dsa.suavizar_dsa_por_bloques_validos(
        dsa=dsa_base,
        mask_no_valida=mask_no_valida_bilateral,
        window=suavizado_bilateral_s,
        min_periods=1,
        center=False
    )

resumen_suavizado_bilateral = pd.DataFrame([
    {
        "matriz": nombre,
        "suavizado_s": suavizado_bilateral_s,
        "filas": len(matriz),
        "filas_no_validas_sqi": int(mask_no_valida_bilateral.sum()),
        "pct_no_valido_sqi": 100 * float(mask_no_valida_bilateral.mean()),
        "db_p50_suav": float(np.nanpercentile(matriz.to_numpy(dtype=float), 50)),
        "db_p95_suav": float(np.nanpercentile(matriz.to_numpy(dtype=float), 95)),
    }
    for nombre, matriz in dsa_bilateral_db_suavizada.items()
])

print("Columnas SQI usadas para la mascara:", sqi_cols_bilateral)
print("Filas no validas por SQI:", int(mask_no_valida_bilateral.sum()), "de", len(mask_no_valida_bilateral))
display(resumen_suavizado_bilateral.round(4))


Columnas SQI usadas para la mascara: ['SQI10', 'SQI10_2', 'SQI10_3', 'SQI10_4']
Filas no validas por SQI: 0 de 5760


,matriz,suavizado_s,filas,filas_no_validas_sqi,pct_no_valido_sqi,db_p50_suav,db_p95_suav
0,izq_13,25,5760,0,0.0,79.3183,99.1728
1,der_24,25,5760,0,0.0,81.1984,94.0498
2,global_1234,25,5760,0,0.0,81.0357,97.5639


In [10]:

# ============================================================
# SEF / MEF exploratorios desde potencia reconstruida
# ============================================================

frecuencias_float = np.asarray(frecuencias_recon, dtype=float)

sef_mef_recon = {}

# Primero calculamos SEF/MEF para cada canal crudo.
for canal, potencia in potencias_por_canal.items():
    sef, mef = fun_dsa.calcular_sef_mef_desde_potencia(
        potencia=potencia,
        frecuencias=frecuencias_float,
        percentil_sef=0.95,
        percentil_mef=0.50
    )
    sef_mef_recon[(f"canal_{canal}", "SEF08")] = sef
    sef_mef_recon[(f"canal_{canal}", "MEDFRQ08")] = mef

# Despues calculamos los agregados fisicos bilaterales.
sef_izq, mef_izq = fun_dsa.calcular_sef_mef_desde_potencia(potencia_izq_13, frecuencias_float)
sef_der, mef_der = fun_dsa.calcular_sef_mef_desde_potencia(potencia_der_24, frecuencias_float)
sef_global, mef_global = fun_dsa.calcular_sef_mef_desde_potencia(potencia_global_1234, frecuencias_float)

sef_mef_recon[("hemisferio_izq_13", "SEF08")] = sef_izq
sef_mef_recon[("hemisferio_izq_13", "MEDFRQ08")] = mef_izq
sef_mef_recon[("hemisferio_der_24", "SEF08")] = sef_der
sef_mef_recon[("hemisferio_der_24", "MEDFRQ08")] = mef_der
sef_mef_recon[("global_1234", "SEF08")] = sef_global
sef_mef_recon[("global_1234", "MEDFRQ08")] = mef_global

print("SEF/MEF reconstruidos para canales, hemisferios fisicos y matriz global.")


SEF/MEF reconstruidos para canales, hemisferios fisicos y matriz global.



## 8. Comparacion exploratoria contra `SEF08` y `MEDFRQ08` del `.spa`

Esta tabla compara variables derivadas de la DSA reconstruida con variables procesadas del `.spa`.

Importante: los resultados no son una validacion definitiva de DSA porque falta `.f_a`. Sirven para ver que columnas del `.spa` tienen datos utiles y si alguna combinacion temporal/canal tiene sentido.


In [11]:

# ============================================================
# Metricas exploratorias SEF/MEF reconstruido vs .spa
# ============================================================

sentinelas_metricas = [-327.7, -3276.0, -3276.8, -3276, -32767, -32768, 32768]

def serie_spa_limpia(columna):
    return (
        pd.to_numeric(df_spa_raw[columna], errors="coerce")
        .replace(sentinelas_metricas, np.nan)
        .to_numpy(dtype=float)
    )


def metricas_serie(a, b, sqi=None):
    """
    Calcula metricas entre dos series temporales.

    Si se pasa SQI, solo se comparan filas con SQI >= 15, siguiendo la idea
    usada en el flujo unilateral para no mezclar periodos de baja calidad.
    """

    n = min(len(a), len(b))
    a = np.asarray(a[:n], dtype=float)
    b = np.asarray(b[:n], dtype=float)

    mask = np.isfinite(a) & np.isfinite(b)

    if sqi is not None:
        sqi = np.asarray(sqi[:n], dtype=float)
        mask = mask & np.isfinite(sqi) & (sqi >= 15)

    if mask.sum() < 3:
        return {
            "n": int(mask.sum()),
            "Pearson": np.nan,
            "Spearman": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
        }

    return {
        "n": int(mask.sum()),
        "Pearson": pearsonr(a[mask], b[mask])[0],
        "Spearman": spearmanr(a[mask], b[mask])[0],
        "MAE": np.mean(np.abs(a[mask] - b[mask])),
        "RMSE": np.sqrt(np.mean((a[mask] - b[mask]) ** 2)),
    }

comparaciones = []

# Comparacion directa por sufijo del .spa.
# Ojo: si una columna del .spa es todo sentinela, n saldra 0.
for canal, sufijo in [(1, ""), (2, "_2"), (3, "_3"), (4, "_4")]:
    sqi_col = "SQI10" + sufijo
    sqi = serie_spa_limpia(sqi_col) if sqi_col in df_spa_raw.columns else None

    for variable in ["SEF08", "MEDFRQ08"]:
        col_spa = variable + sufijo
        if col_spa not in df_spa_raw.columns:
            continue

        met = metricas_serie(
            sef_mef_recon[(f"canal_{canal}", variable)],
            serie_spa_limpia(col_spa),
            sqi=sqi
        )

        comparaciones.append({
            "comparacion": f"recon_canal_{canal}_vs_spa_{col_spa}",
            "tipo": "canal_crudo_vs_sufijo_spa",
            "variable": variable,
            **met
        })

# Comparacion por hemisferio fisico contra promedios de columnas SPA.
# Se mantiene como exploracion, no como decision final.
hemisferios = [
    ("hemisferio_izq_13", ["", "_3"], ["SQI10", "SQI10_3"]),
    ("hemisferio_der_24", ["_2", "_4"], ["SQI10_2", "SQI10_4"]),
]

for hemi, sufijos, sqi_cols in hemisferios:
    sqi_stack = np.vstack([serie_spa_limpia(c) for c in sqi_cols if c in df_spa_raw.columns])
    sqi_prom = np.nanmean(sqi_stack, axis=0) if len(sqi_stack) else None

    for variable in ["SEF08", "MEDFRQ08"]:
        cols = [variable + suf for suf in sufijos if variable + suf in df_spa_raw.columns]
        if not cols:
            continue

        spa_stack = np.vstack([serie_spa_limpia(c) for c in cols])
        spa_prom = np.nanmean(spa_stack, axis=0)

        met = metricas_serie(
            sef_mef_recon[(hemi, variable)],
            spa_prom,
            sqi=sqi_prom
        )

        comparaciones.append({
            "comparacion": f"recon_{hemi}_vs_promedio_spa_{variable}",
            "tipo": "hemisferio_crudo_vs_promedio_spa",
            "variable": variable,
            **met
        })

df_metricas_exploratorias = pd.DataFrame(comparaciones)
display(df_metricas_exploratorias.round(4))


,comparacion,tipo,variable,n,Pearson,Spearman,MAE,RMSE
0,recon_canal_1_vs_spa_SEF08,canal_crudo_vs_sufijo_spa,SEF08,5760,0.1718,0.1532,8.1855,9.4119
1,recon_canal_1_vs_spa_MEDFRQ08,canal_crudo_vs_sufijo_spa,MEDFRQ08,5760,0.0419,0.0502,2.0449,2.6429
2,recon_canal_2_vs_spa_SEF08_2,canal_crudo_vs_sufijo_spa,SEF08,0,NaN,NaN,NaN,NaN
3,recon_canal_2_vs_spa_MEDFRQ08_2,canal_crudo_vs_sufijo_spa,MEDFRQ08,0,NaN,NaN,NaN,NaN
4,recon_canal_3_vs_spa_SEF08_3,canal_crudo_vs_sufijo_spa,SEF08,5760,0.2037,0.1935,7.7610,8.9832
5,recon_canal_3_vs_spa_MEDFRQ08_3,canal_crudo_vs_sufijo_spa,MEDFRQ08,5760,0.0519,0.0923,2.1034,2.7748
6,recon_canal_4_vs_spa_SEF08_4,canal_crudo_vs_sufijo_spa,SEF08,0,NaN,NaN,NaN,NaN
7,recon_canal_4_vs_spa_MEDFRQ08_4,canal_crudo_vs_sufijo_spa,MEDFRQ08,0,NaN,NaN,NaN,NaN
8,recon_hemisferio_izq_13_vs_promedio_spa_SEF08,hemisferio_crudo_vs_promedio_spa,SEF08,5760,0.1993,0.1833,7.9774,9.1526
9,recon_hemisferio_izq_13_vs_promedio_spa_MEDFRQ08,hemisferio_crudo_vs_promedio_spa,MEDFRQ08,5760,0.0537,0.0907,2.0637,2.5913


## 9. Asimetria reconstruida de mejor esfuerzo

`ASYM09` aparece como una variable comun/interhemisferica. Como no conocemos la formula propietaria del BIS, se calculan proxies transparentes desde la potencia total reconstruida:

- `izq_pct`: porcentaje de potencia reconstruida en el lado izquierdo;
- `der_pct`: porcentaje de potencia reconstruida en el lado derecho;
- `diff_pct`: diferencia porcentual izquierda menos derecha;
- `log_ratio_db`: ratio logaritmico izquierda/derecha.

Estas variables no se presentan como equivalentes exactos a `ASYM09`; solo sirven para ver si el balance energetico reconstruido se mueve de forma parecida a la asimetria exportada.

In [12]:

# ============================================================
# Proxies de asimetria bilateral reconstruida
# ============================================================

# El helper acepta matrices tiempo x frecuencia y suma la potencia por fila.
df_asimetria_recon = fun_dsa_b.calcular_asimetrias_bilaterales(
    potencia_izq=potencia_izq_13,
    potencia_der=potencia_der_24
)
df_asimetria_recon.insert(0, "tiempo_s", resultado_recon_bilateral["tiempo_s"])

asim_spa = serie_spa_limpia("ASYM09") if "ASYM09" in df_spa_raw.columns else None
sqi_asim = serie_spa_limpia("SQI10") if "SQI10" in df_spa_raw.columns else None

comparaciones_asimetria = []

if asim_spa is not None:
    for proxy in ["izq_pct", "der_pct", "diff_pct", "log_ratio_db"]:
        met = metricas_serie(
            df_asimetria_recon[proxy].to_numpy(dtype=float),
            asim_spa,
            sqi=sqi_asim
        )

        comparaciones_asimetria.append({
            "proxy_reconstruido": proxy,
            "referencia_spa": "ASYM09",
            **met
        })

df_metricas_asimetria = pd.DataFrame(comparaciones_asimetria)

if len(df_metricas_asimetria):
    df_metricas_asimetria["abs_Pearson"] = df_metricas_asimetria["Pearson"].abs()
    df_metricas_asimetria = df_metricas_asimetria.sort_values(
        ["abs_Pearson", "Spearman"],
        ascending=[False, False]
    ).reset_index(drop=True)

    mejor_proxy_asimetria = df_metricas_asimetria.iloc[0].to_dict()

    display(df_metricas_asimetria.round(4))
    print(
        "Mejor proxy por |Pearson|:",
        mejor_proxy_asimetria["proxy_reconstruido"],
        "Pearson=", round(mejor_proxy_asimetria["Pearson"], 4),
        "Spearman=", round(mejor_proxy_asimetria["Spearman"], 4),
        "n=", int(mejor_proxy_asimetria["n"])
    )
else:
    mejor_proxy_asimetria = None
    print("No se pudo comparar contra ASYM09 porque no hay columna valida disponible.")

print("Resumen descriptivo de proxies reconstruidos:")
display(df_asimetria_recon[["izq_pct", "der_pct", "diff_pct", "log_ratio_db"]].describe().T.round(4))


,proxy_reconstruido,referencia_spa,n,Pearson,Spearman,MAE,RMSE,abs_Pearson
0,log_ratio_db,ASYM09,5760,0.0021,-0.004,47.3754,48.0223,0.0021
1,diff_pct,ASYM09,5760,0.0003,-0.004,22.6061,29.3202,0.0003
2,izq_pct,ASYM09,5760,0.0003,-0.004,19.0806,22.3108,0.0003
3,der_pct,ASYM09,5760,-0.0003,0.004,20.7255,23.9537,0.0003


Mejor proxy por |Pearson|: log_ratio_db Pearson= 0.0021 Spearman= -0.004 n= 5760
Resumen descriptivo de proxies reconstruidos:


,count,mean,std,min,25%,50%,75%,max
izq_pct,5760.0,68.2527,12.2075,7.5167,61.8261,68.9416,75.2150,99.9826
der_pct,5760.0,31.7473,12.2075,0.0174,24.7850,31.0584,38.1739,92.4833
diff_pct,5760.0,36.5055,24.4150,-84.9666,23.6523,37.8832,50.4301,99.9652
log_ratio_db,5760.0,3.6644,3.0458,-10.9003,2.0941,3.4630,4.8212,37.5955


## 10. Conclusiones de esta primera exploracion

Este cierre deja por escrito lo que se puede afirmar con los archivos actuales y lo que queda como limitacion.

La diferencia respecto a la primera version de la exploracion es importante: el `.f_a`/GLF vacio ya no se trata como un bloqueo para trabajar, sino como una limitacion de validacion directa. Aun asi, la reconstruccion bilateral desde `.r4a` queda generada y documentada.

In [13]:

# ============================================================
# Resumen de hallazgos y limitaciones
# ============================================================

fila_asimetria = {
    "tema": "Asimetria reconstruida",
    "resultado": "Se calcularon proxies izq_pct, der_pct, diff_pct y log_ratio_db desde potencia izquierda/derecha.",
    "estado": "exploratorio; no equivale a formula propietaria ASYM09",
}

if 'mejor_proxy_asimetria' in globals() and mejor_proxy_asimetria is not None:
    fila_asimetria["resultado"] = (
        f"Mejor proxy frente a ASYM09 por |Pearson|: {mejor_proxy_asimetria['proxy_reconstruido']} "
        f"(Pearson={mejor_proxy_asimetria['Pearson']:.4f}, Spearman={mejor_proxy_asimetria['Spearman']:.4f}, "
        f"n={int(mejor_proxy_asimetria['n'])})."
    )

resumen_exploracion = pd.DataFrame([
    {
        "tema": "Caso bilateral disponible",
        "resultado": "L04141330 contiene .r4a de cuatro canales y .spa procesado.",
        "estado": "usable para inventario y reconstruccion desde crudo",
    },
    {
        "tema": "Referencia .f_a / GLF",
        "resultado": "L04141330.f_a pesa 0 bytes; el usuario confirma que era esperable.",
        "estado": "no bloquea reconstruccion, pero impide comparacion DSA matriz-a-matriz",
    },
    {
        "tema": "Reconstruccion mejor esfuerzo",
        "resultado": "Se reconstruyen canales 1-4 con Welch 1 s; izq=media lineal 1/3; der=media lineal 2/4; global=media lineal 1/2/3/4.",
        "estado": "implementado en helper reutilizable",
    },
    {
        "tema": "Suavizado bilateral",
        "resultado": "Se genera salida suavizada causal de 25 s para izq_13, der_24 y global_1234 respetando mascara SQI.",
        "estado": "exploratorio; sin shift porque no hay referencia .f_a",
    },
    {
        "tema": "Columnas .spa",
        "resultado": "Variables espectrales principales son informativas en base y _3; _2 y _4 son sentinelas para muchas de ellas.",
        "estado": "requiere decision de mapeo antes de refactorizar helpers SPA",
    },
    {
        "tema": "Canales crudos .r4a",
        "resultado": "Se leen cuatro canales: canal_1_uV, canal_2_uV, canal_3_uV, canal_4_uV.",
        "estado": "compatible con hipotesis 1/3 izquierdo y 2/4 derecho",
    },
    {
        "tema": "Asimetria SPA",
        "resultado": "ASYM09 base contiene datos; ASYM09_2, ASYM09_3 y ASYM09_4 aparecen como sentinelas en este caso.",
        "estado": "tratar como variable comun/interhemisferica",
    },
    fila_asimetria,
    {
        "tema": "Siguiente paso",
        "resultado": "Probar la misma reconstruccion en mas bilaterales y, si aparece un .f_a no vacio, validar matriz-a-matriz.",
        "estado": "pendiente",
    },
])

display(resumen_exploracion)


,tema,resultado,estado
0,Caso bilateral disponible,L04141330 contiene .r4a de cuatro canales y .s...,usable para inventario y reconstruccion desde ...
1,Referencia .f_a / GLF,L04141330.f_a pesa 0 bytes; el usuario confirm...,"no bloquea reconstruccion, pero impide compara..."
2,Reconstruccion mejor esfuerzo,Se reconstruyen canales 1-4 con Welch 1 s; izq...,implementado en helper reutilizable
3,Suavizado bilateral,Se genera salida suavizada causal de 25 s para...,exploratorio; sin shift porque no hay referenc...
4,Columnas .spa,Variables espectrales principales son informat...,requiere decision de mapeo antes de refactoriz...
5,Canales crudos .r4a,"Se leen cuatro canales: canal_1_uV, canal_2_uV...",compatible con hipotesis 1/3 izquierdo y 2/4 d...
6,Asimetria SPA,"ASYM09 base contiene datos; ASYM09_2, ASYM09_3...",tratar como variable comun/interhemisferica
7,Asimetria reconstruida,Mejor proxy frente a ASYM09 por |Pearson|: log...,exploratorio; no equivale a formula propietari...
8,Siguiente paso,Probar la misma reconstruccion en mas bilatera...,pendiente
